# Chess Neural Network — Training Pipeline

Auto-detects training phase based on available checkpoints.
Runs on Kaggle T4/P100 GPU.

In [ ]:
# Clone the repo and install dependencies
!git clone https://github.com/amiohossain/Chess.git /kaggle/working/Chess
!pip install -q -r /kaggle/working/Chess/requirements.txt
!pip install -q zstandard  # For zstd decompression fallback

In [ ]:
import os, sys, glob, subprocess, json
sys.path.insert(0, '/kaggle/working/Chess')
os.makedirs('/kaggle/working/data', exist_ok=True)

# Check for existing data
HDF5_PATH = '/kaggle/working/supervised_positions.h5'
h5_input = glob.glob('/kaggle/input/chess-training-data/**/*.h5', recursive=True)

if h5_input:
    print(f'Using existing HDF5: {h5_input[0]}')
    !cp {h5_input[0]} {HDF5_PATH}
else:
    # Decompress and process PGN
    zst_files = sorted(glob.glob('/kaggle/input/chess-training-data/**/*.pgn.zst', recursive=True))
    print(f'Found {len(zst_files)} zst files')
    
    if not zst_files:
        zst_files = sorted(glob.glob('/kaggle/input/chess-training-data/**/*.zst', recursive=True))
        print(f'Found {len(zst_files)} .zst files (non-specific)')
    
    if zst_files:
        combined_pgn = '/kaggle/working/combined.pgn'
        for i, zst in enumerate(zst_files):
            pgn_part = f'/kaggle/working/temp_{i}.pgn'
            print(f'[{i+1}/{len(zst_files)}] Decompressing {os.path.basename(zst)}...')
            try:
                subprocess.run(['zstd', '-d', '-f', zst, '-o', pgn_part], check=True, capture_output=True)
            except:
                import zstandard as zstd_lib
                with open(zst, 'rb') as src, open(pgn_part, 'wb') as dst:
                    zstd_lib.ZstdDecompressor().copy_stream(src, dst)
            
            with open(pgn_part, 'r', errors='replace') as src:
                with open(combined_pgn, 'a' if i > 0 else 'w') as dst:
                    dst.write(src.read())
            os.remove(pgn_part)
            print(f'  -> {os.path.getsize(combined_pgn)/1e6:.0f} MB so far')

        # Process to HDF5 (no Elo filter for testing)
        from src.data.pgn_processor import process_pgn
        n = process_pgn(pgn_path=combined_pgn, output_path=HDF5_PATH, max_positions=5_000_000)
        os.remove(combined_pgn)
        print(f'HDF5 created: {n} positions')
    else:
        print('No data files found at all!')
        !ls -la /kaggle/input/chess-training-data/

# Verify HDF5 exists
if os.path.exists(HDF5_PATH):
    import h5py
    with h5py.File(HDF5_PATH, 'r') as f:
        n = f.attrs['num_positions']
    print(f'Training data ready: {n} positions in {HDF5_PATH}')
else:
    raise FileNotFoundError(f'{HDF5_PATH} not found — data processing failed')

In [ ]:
# Run the training pipeline (auto-detects phase)
from src.kaggle.kaggle_main import run_session
run_session()